In [66]:
# basic libraries
import pandas as pd
import numpy as np
import shap
import pickle
import matplotlib.pyplot as plt
import os

In [67]:
# load test data
X_test = pd.read_csv("/content/X_test_clean.csv")

# check shape
print("shape:", X_test.shape)
X_test.head()

shape: (1126, 18)


,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,PreferredLoginDevice,PreferredPaymentMode,Gender,PreferedOrderCat,MaritalStatus
0,17.0,3.0,9.0,4.0,4.0,1.0,3.0,0.0,21.0,6.0,8.0,8.0,181.75,Computer,Debit Card,Male,Laptop & Accessory,Married
1,26.0,3.0,13.0,3.0,4.0,1.0,8.0,1.0,18.0,0.0,1.0,3.0,209.34,Mobile Phone,E-Wallet,Female,Fashion,Married
2,5.0,1.0,15.0,4.0,4.0,1.0,3.0,0.0,24.0,2.0,2.0,9.0,169.24,Mobile Phone,Debit Card,Female,Laptop & Accessory,Single
3,25.0,1.0,8.0,2.0,4.0,2.0,8.0,0.0,15.0,0.0,2.0,2.0,132.12,Mobile Phone,Debit Card,Female,Mobile Phone,Single
4,4.0,3.0,8.0,2.0,3.0,3.0,2.0,1.0,19.0,8.0,9.0,6.0,198.52,Computer,Debit Card,Female,Fashion,Married


In [68]:
# categorical columns (must match notebook 05 EXACTLY)
categorical_cols = [
    'PreferredLoginDevice',
    'PreferredPaymentMode',
    'Gender',
    'PreferedOrderCat',
    'MaritalStatus'
]

# apply one-hot encoding
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

print("After encoding:", X_test.shape)

After encoding: (1126, 25)


In [69]:
# load LightGBM model
with open("/content/final_lgb_model.pkl", "rb") as f:
    lgb_model = pickle.load(f)

# load CatBoost model
with open("/content/final_cb_model.pkl", "rb") as f:
    cb_model = pickle.load(f)

In [70]:
# get feature names used in training
expected_cols = lgb_model.feature_name_

# add missing columns in test
for col in expected_cols:
    if col not in X_test.columns:
        X_test[col] = 0

# remove extra columns (if any)
X_test = X_test[expected_cols]

print("Final shape:", X_test.shape)

Final shape: (1126, 25)


In [72]:
# create explainers
explainer_lgb = shap.TreeExplainer(lgb_model)
explainer_cb = shap.TreeExplainer(cb_model)

In [74]:
# compute SHAP values
shap_values_lgb = explainer_lgb.shap_values(X_test)

# Create a copy of X_test for CatBoost to avoid modifying the input for LightGBM
X_test_cb = X_test.copy()

# CatBoost model expects feature names with spaces in one-hot encoded categories
# while pd.get_dummies converts spaces to underscores. Renaming columns in X_test_cb
# to match CatBoost's expected feature names.
renamed_cb_columns = []
for col_name in X_test_cb.columns:
    new_col_name = col_name
    # Specific replacements based on the error message and common categorical values
    if 'PreferredLoginDevice_Mobile_Phone' in col_name:
        new_col_name = new_col_name.replace('PreferredLoginDevice_Mobile_Phone', 'PreferredLoginDevice_Mobile Phone')
    if 'PreferredPaymentMode_Credit_Card' in col_name:
        new_col_name = new_col_name.replace('PreferredPaymentMode_Credit_Card', 'PreferredPaymentMode_Credit Card')
    if 'PreferredPaymentMode_Debit_Card' in col_name:
        new_col_name = new_col_name.replace('PreferredPaymentMode_Debit_Card', 'PreferredPaymentMode_Debit Card')
    if 'PreferredPaymentMode_Cash_on_Delivery' in col_name:
        new_col_name = new_col_name.replace('PreferredPaymentMode_Cash_on_Delivery', 'PreferredPaymentMode_Cash on Delivery')
    if 'PreferedOrderCat_Laptop_&_Accessory' in col_name:
        new_col_name = new_col_name.replace('PreferedOrderCat_Laptop_&_Accessory', 'PreferedOrderCat_Laptop & Accessory')
    if 'PreferedOrderCat_Mobile_Phone' in col_name:
        new_col_name = new_col_name.replace('PreferedOrderCat_Mobile_Phone', 'PreferedOrderCat_Mobile Phone')
    renamed_cb_columns.append(new_col_name)
X_test_cb.columns = renamed_cb_columns

# Now compute SHAP values for CatBoost with the renamed DataFrame
shap_values_cb = explainer_cb.shap_values(X_test_cb)

# fix for binary classification (sometimes returns list)
if isinstance(shap_values_lgb, list):
    shap_values_lgb = shap_values_lgb[1]

if isinstance(shap_values_cb, list):
    shap_values_cb = shap_values_cb[1]

print("SHAP computed")

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


SHAP computed


In [75]:
# SHAP SUMMARY PLOT - LIGHTGBM
plt.figure()

shap.summary_plot(
    shap_values_lgb,
    X_test,
    show=False
)

plt.savefig("/content/shap_summary_lgb.png", dpi=150)
plt.close()

In [76]:
# SHAP SUMMARY PLOT - CATBOOST
plt.figure()

shap.summary_plot(
    shap_values_cb,
    X_test,
    show=False
)

plt.savefig("/content/shap_summary_cb.png", dpi=150)
plt.close()

In [77]:
# EXTRACT TOP FEATURES
def get_top_features(shap_values, X, top_n=10):

    # mean absolute SHAP value
    importance = np.abs(shap_values).mean(axis=0)

    df = pd.DataFrame({
        "feature": X.columns,
        "importance": importance
    })

    df = df.sort_values(by="importance", ascending=False)

    return df.head(top_n)

# get top features
top_lgb = get_top_features(shap_values_lgb, X_test)
top_cb = get_top_features(shap_values_cb, X_test)

print("Top LightGBM:\n", top_lgb)
print("\nTop CatBoost:\n", top_cb)

Top LightGBM:
                                 feature  importance
0                                Tenure    1.302976
7                              Complain    0.648959
6                       NumberOfAddress    0.424278
12                       CashbackAmount    0.382666
5                     SatisfactionScore    0.334395
1                              CityTier    0.275902
11                    DaySinceLastOrder    0.243782
24                 MaritalStatus_Single    0.240636
2                       WarehouseToHome    0.239835
20  PreferedOrderCat_Laptop_&_Accessory    0.232359

Top CatBoost:
                               feature  importance
0                              Tenure    1.767752
7                            Complain    1.097476
24               MaritalStatus_Single    0.576735
6                     NumberOfAddress    0.552351
1                            CityTier    0.540589
11                  DaySinceLastOrder    0.487531
5                   SatisfactionScore    0.4104